# Merge LoRA -> convert to MLX -> verify -> push to HuggingFace

Same pipeline as `merge_and_export_gguf.ipynb`, but the output is **MLX**
(Apple's format) instead of GGUF. Goal: MLX uses the HF-faithful Idefics3
image preprocessing (`processing_idefics3`), unlike llama.cpp's `clip.cpp` —
so on-device quality should match the vLLM serving output (which llama.cpp/
PocketPal did not, on dense headers).

Steps:
1. Pull the **Production** adapter from MLflow (same keys as the GGUF notebook).
2. **Merge** it into the base in PyTorch (bf16).
3. **Convert** the merged HF model to MLX (`mlx_vlm.convert`, no quantization).
4. **VERIFY** with `mlx_vlm.generate` on the test page — compare with serving
   *before* doing any iOS work.
5. **Push** the MLX folder to HuggingFace.

> **Where to run:** MLX needs Apple Silicon (Mac) OR a Linux box with the mlx
> CUDA/CPU build (Colab). It does **not** run on Windows. Convert/verify are not
> heavy — CPU mlx is fine (just slower).

> **iPhone:** the resulting MLX model runs on iPhone via **`ml-explore/mlx-swift-lm`**
> (`MLXVLM`, which supports `idefics3`) — a separate mlx-swift app (adapt the
> `VLMEval` example). NOT PocketPal (that is llama.cpp). Build that only after the
> verify step below confirms MLX matches serving.

## 0. Install

In [ ]:
!pip -q install -U mlx-vlm transformers peft huggingface_hub mlflow boto3 sentencepiece pillow
# If `import mlx` fails on Colab (Linux+GPU), install the CUDA build instead:
#   !pip -q install -U "mlx[cuda]" mlx-vlm
# On Apple Silicon Mac the default wheel works as-is.
import mlx_vlm, mlx.core as mx
print('mlx-vlm ok; default device:', mx.default_device())

## 1. Config (keys identical to the GGUF notebook)

In [ ]:
import os

BASE_MODEL = 'ibm-granite/granite-docling-258M'
MODEL_NAME = 'granite-docling-adapter'              # MLflow registered model
HF_TARGET  = 'nbdaaa/granite-docling-258M-mine-mlx' # MLX repo to push to
MERGED_DIR = 'granite-docling-merged'
MLX_DIR    = 'granite-docling-mlx'
IMG        = 'ttcp1.jpg'   # upload this test page to the session for the verify step

VM = '34.142.198.19'       # GCP VM public IP (DYNAMIC — set the current one)
os.environ['MLFLOW_TRACKING_URI']    = f'http://{VM}:5000'
os.environ['MLFLOW_S3_ENDPOINT_URL'] = f'http://{VM}:9000'   # MinIO API (9000)
os.environ['AWS_ACCESS_KEY_ID']      = 'minioadmin'
os.environ['AWS_SECRET_ACCESS_KEY']  = 'Nbda__1002'

## 2. HuggingFace login (write token, for the push)

In [ ]:
from huggingface_hub import login
login()  # paste a token with WRITE access

## 3. Pull the Production adapter from MLflow

In [ ]:
import mlflow
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
client = mlflow.MlflowClient()

run_id = None
for v in client.search_model_versions(f"name='{MODEL_NAME}'"):
    if v.current_stage == 'Production':
        run_id = v.run_id
        print('Production = version', v.version, 'run', run_id)
        break
assert run_id, 'No Production version found'

ADAPTER_DIR = client.download_artifacts(run_id, 'adapter', 'adapter_dl')
print('adapter at:', ADAPTER_DIR, os.listdir(ADAPTER_DIR))

## 4. Merge LoRA into the base (PyTorch, bf16)

In [ ]:
import os
# Don't let transformers pull TensorFlow (protobuf clash on Colab).
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'

import torch
from transformers import AutoModelForImageTextToText
from peft import PeftModel

base = AutoModelForImageTextToText.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model = model.merge_and_unload()
model.save_pretrained(MERGED_DIR, safe_serialization=True)
print('merged ->', MERGED_DIR)
print(os.listdir(MERGED_DIR))

In [ ]:
# Copy tokenizer + processor config from the base (AutoProcessor.save errors on
# this transformers build; keep the merged config.json).
import os, shutil
from huggingface_hub import snapshot_download

src = snapshot_download(
    BASE_MODEL,
    allow_patterns=['*.json', '*.txt', '*.model', 'tokenizer*',
                    'preprocessor_config.json', 'processor_config.json',
                    'chat_template*', 'merges.txt', 'vocab.json',
                    'special_tokens_map.json', 'added_tokens.json'],
)
for f in os.listdir(src):
    s = os.path.join(src, f)
    if os.path.isfile(s) and f != 'config.json':
        shutil.copy(s, os.path.join(MERGED_DIR, f))
print(sorted(os.listdir(MERGED_DIR)))

## 5. Convert merged HF model -> MLX (no quantization)

In [ ]:
# No -q / --quantize  -> keep full precision (idefics3 loader).
!python -m mlx_vlm.convert --hf-path {MERGED_DIR} --mlx-path {MLX_DIR}
!ls -la {MLX_DIR}

## 6. VERIFY (do this before any iOS work)
Upload the test page as `ttcp1.jpg` first. Compare the output with the serving
result: the dense top headers (ỦY BAN NHÂN DÂN / CỘNG HÒA… / Chủ tịch / signature)
should now be clean if MLX's preprocessing matches HF.

In [ ]:
!python -m mlx_vlm.generate --model {MLX_DIR} --image {IMG} \
  --prompt "Convert this page to docling format." \
  --temperature 0.0 --max-tokens 4096

## 7. Push the MLX folder to HuggingFace

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(HF_TARGET, repo_type='model', exist_ok=True)
api.upload_folder(folder_path=MLX_DIR, repo_id=HF_TARGET)
print('pushed ->', 'https://huggingface.co/' + HF_TARGET)

## Next: run on iPhone (only if VERIFY matched serving)
- The MLX model runs on iPhone via **`ml-explore/mlx-swift-lm`** (`MLXVLM`, `idefics3`).
- Build/adapt the **`VLMEval`** mlx-swift example into a small app that downloads
  `HF_TARGET`, takes an image + the prompt `Convert this page to docling format.`,
  and shows the DocTags. Build with Xcode (Mac) or Codemagic.
- This is NOT PocketPal (PocketPal is llama.cpp and cannot load MLX).

**Notes:** no quantization (full precision). If `mlx_vlm.convert` rejects the
granite-docling config, check that `config.json` has `model_type: idefics3`; the
merged dir keeps the base config which already does.